# Phase 5: Inference Pipeline & Web UI

Project : MBTI Personality Type Prediction from Text
Models  : Logistic Regression + SMOTE (winner from Phase 4)

What this phase does:
---
Turns our trained models into a usable prediction system.

1. MODEL TRAINING & PERSISTENCE
- Re-trains the 4 winning LR+SMOTE binary classifiers (fast, <30s)
- Saves them alongside the fitted TF-IDF vectorizer to disk
- On subsequent runs, loads from disk (no re-training needed)

2. predict_mbti(text) FUNCTION
- Takes a raw string (comment, paragraph, essay, etc.)
- Cleans it using Phase 2's preprocessing pipeline
- Vectorizes it with the saved TF-IDF vectorizer
- Runs it through 4 binary classifiers → 4-letter MBTI type

3. GRADIO WEB UI
- Simple, elegant interface for interactive predictions
- User types/pastes text → sees predicted MBTI type + description
- Includes confidence scores for each axis

Why Logistic Regression?
---
Phase 4 showed that LR+SMOTE beat both XGBoost and LightGBM on
macro-F1 across ALL 4 axes.  On sparse high-dimensional TF-IDF
features (~5000 features, ~7K samples), a well-regularised linear
model with proper class balancing is hard to beat.

LR+SMOTE avg macro-F1 : 0.6829
XGBoost  avg macro-F1 : 0.5637
LightGBM avg macro-F1 : 0.6006


In [1]:
%matplotlib inline

## Preprocessing Function (from Phase 2)

The `preprocess_text` function and its dependencies are inlined here
so this notebook is self-contained.


In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure NLTK data is available
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

# --- MBTI Leak Prevention Constants ---
MBTI_TYPES = [
    "infp", "infj", "intp", "intj",
    "isfp", "isfj", "istp", "istj",
    "enfp", "enfj", "entp", "entj",
    "esfp", "esfj", "estp", "estj",
]

COGNITIVE_FUNCTIONS = [
    "ni", "ne", "si", "se",
    "ti", "te", "fi", "fe",
]

TEMPERAMENT_GROUPS = [
    "nt", "nf", "sf", "st",
    "sj", "sp", "nj", "np",
]

MBTI_WILDCARDS = [
    "xnfp", "xnfj", "xntp", "xntj",
    "xsfp", "xsfj", "xstp", "xstj",
    "xnfx", "xntx", "xsfx", "xstx",
]

MBTI_META = ["mbti", "myers", "briggs", "jungian", "enneagram", "socionics", "sx", "so"]

_mbti_with_suffix = [t + r"(?:s|'s)?" for t in MBTI_TYPES]
_all_leak_tokens = (
    _mbti_with_suffix + COGNITIVE_FUNCTIONS + TEMPERAMENT_GROUPS
    + MBTI_WILDCARDS + MBTI_META
)
MBTI_PATTERN = re.compile(
    r"\b(" + "|".join(_all_leak_tokens) + r")\b", flags=re.IGNORECASE
)

URL_PATTERN      = re.compile(r"https?://\S+|www\.\S+")
EMAIL_PATTERN    = re.compile(r"\S+@\S+\.\S+")
NON_ALPHA_PATTERN = re.compile(r"[^a-z\s]")
MULTI_SPACE       = re.compile(r"\s+")

URL_RESIDUE = {"http", "https", "www", "com", "org", "net", "edu", "gov",
               "html", "htm", "php", "asp", "aspx", "jpg", "jpeg", "png",
               "gif", "youtube", "imgur", "tumblr", "wordpress", "blogspot"}
STOP_WORDS = set(stopwords.words("english")) | URL_RESIDUE

MBTI_SET = (
    set(MBTI_TYPES) | {t + "s" for t in MBTI_TYPES}
    | set(COGNITIVE_FUNCTIONS) | set(TEMPERAMENT_GROUPS)
    | set(MBTI_WILDCARDS) | set(MBTI_META)
)

LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text: str) -> str:
    """Clean a single user's concatenated posts string."""
    text = text.replace("|||", " ")
    text = text.lower()
    text = URL_PATTERN.sub("", text)
    text = EMAIL_PATTERN.sub("", text)
    text = MBTI_PATTERN.sub("", text)
    text = NON_ALPHA_PATTERN.sub(" ", text)
    text = MBTI_PATTERN.sub("", text)
    text = MULTI_SPACE.sub(" ", text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and t not in MBTI_SET and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if t not in MBTI_SET]
    return " ".join(tokens)

print("Preprocessing function loaded successfully.")

Preprocessing function loaded successfully.


In [3]:
# -- Imports ------------------------------------------------------------------
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

# Import the preprocessing function from Phase 2
# This ensures we use the EXACT same cleaning pipeline at inference time
# as we did during training — critical for consistency.
# preprocess_text is defined below (inlined from phase2_preprocessing.py)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# -- Configuration ------------------------------------------------------------

# Directory to save/load trained models and vectorizer
BASE_DIR = os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, "models")

# Paths for saved artifacts
TFIDF_PATH = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
MODELS_PATH = os.path.join(MODEL_DIR, "lr_models.pkl")         # dict of 4 models
METADATA_PATH = os.path.join(MODEL_DIR, "training_metadata.pkl")

# Dataset for training (if models haven't been saved yet)
DATA_PATH = os.path.join(BASE_DIR, "mbti_cleaned.csv")

# Must match Phase 3/4 exactly for reproducibility
MAX_FEATURES = 5000
MAX_DF = 0.95
MIN_DF = 2
NGRAM_RANGE = (1, 2)
RANDOM_STATE = 42

# The 4 MBTI axes — each tuple is (name, position, label_0, label_1)
# label_0 maps to prediction=0,  label_1 maps to prediction=1
MBTI_AXES = [
    ("E_I", 0, "E", "I"),
    ("S_N", 1, "S", "N"),
    ("T_F", 2, "T", "F"),
    ("J_P", 3, "J", "P"),
]

# MBTI type descriptions for the UI
# Brief descriptions of each of the 16 personality types
MBTI_DESCRIPTIONS = {
    "INTJ": "The Architect - Imaginative and strategic thinkers, with a plan for everything.",
    "INTP": "The Logician - Innovative inventors with an unquenchable thirst for knowledge.",
    "ENTJ": "The Commander - Bold, imaginative, and strong-willed leaders.",
    "ENTP": "The Debater - Smart and curious thinkers who enjoy intellectual challenges.",
    "INFJ": "The Advocate - Quiet and mystical, yet very inspiring and tireless idealists.",
    "INFP": "The Mediator - Poetic, kind, and altruistic, always eager to help a good cause.",
    "ENFJ": "The Protagonist - Charismatic and inspiring leaders who mesmerise their listeners.",
    "ENFP": "The Campaigner - Enthusiastic, creative, and sociable free spirits.",
    "ISTJ": "The Logistician - Practical and fact-minded, whose reliability cannot be doubted.",
    "ISFJ": "The Defender - Very dedicated and warm protectors, always ready to defend loved ones.",
    "ESTJ": "The Executive - Excellent administrators, unsurpassed at managing things or people.",
    "ESFJ": "The Consul - Extraordinarily caring, social, and popular, always eager to help.",
    "ISTP": "The Virtuoso - Bold and practical experimenters, masters of all kinds of tools.",
    "ISFP": "The Adventurer - Flexible and charming artists, always ready to explore something new.",
    "ESTP": "The Entrepreneur - Smart, energetic, and very perceptive, living on the edge.",
    "ESFP": "The Entertainer - Spontaneous, energetic, and enthusiastic, life is never boring around them.",
}

## 1. TRAIN & SAVE MODELS


In [4]:
def train_and_save_models():
    """
    Train the 4 winning LR+SMOTE classifiers and save them to disk.

    This function:
        1. Loads mbti_cleaned.csv
        2. Performs the same 80/20 stratified split used in Phase 3/4
        3. Fits the TF-IDF vectorizer on training text
        4. For each MBTI axis:
           a. Applies SMOTE to the training data
           b. Trains a Logistic Regression with class_weight='balanced'
           c. Evaluates on the held-out test set
        5. Saves the TF-IDF vectorizer and all 4 models to the models/ directory

    Returns
    -------
    tfidf : TfidfVectorizer
        Fitted vectorizer.
    models : dict
        {axis_name: fitted LogisticRegression model}
    """
    print("=" * 60)
    print("  PHASE 5: TRAINING & SAVING INFERENCE MODELS")
    print("=" * 60)

    # -- Load data ------------------------------------------------------------
    df = pd.read_csv(DATA_PATH)
    df = df.dropna(subset=["clean_posts"])
    df = df[df["clean_posts"].str.strip() != ""]
    df = df.reset_index(drop=True)
    print(f"  Loaded {len(df)} rows from mbti_cleaned.csv")

    # -- Create binary targets ------------------------------------------------
    for axis_name, pos, label_0, label_1 in MBTI_AXES:
        df[axis_name] = (df["type"].str[pos] == label_1).astype(int)

    # -- Train/test split (same seed and strategy as Phase 3/4) ---------------
    X_text = df["clean_posts"]
    y_full = df["type"]

    X_train_text, X_test_text, train_idx, test_idx = train_test_split(
        X_text, df.index, test_size=0.20,
        random_state=RANDOM_STATE, stratify=y_full,
    )
    print(f"  Train: {len(X_train_text)} | Test: {len(X_test_text)}")

    # -- TF-IDF vectorization -------------------------------------------------
    tfidf = TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        max_df=MAX_DF,
        min_df=MIN_DF,
        sublinear_tf=True,
        strip_accents="unicode",
    )
    X_train_tfidf = tfidf.fit_transform(X_train_text)
    X_test_tfidf = tfidf.transform(X_test_text)
    print(f"  TF-IDF: {X_train_tfidf.shape[1]} features")

    # -- Train 4 LR+SMOTE models ---------------------------------------------
    models = {}
    metadata = {"axes": MBTI_AXES, "scores": {}}

    for axis_name, pos, label_0, label_1 in MBTI_AXES:
        y_train = df.loc[train_idx, axis_name].values
        y_test = df.loc[test_idx, axis_name].values

        # Apply SMOTE to training data only
        smote = SMOTE(random_state=RANDOM_STATE)
        X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)

        # Train Logistic Regression (same config as Phase 4)
        model = LogisticRegression(
            max_iter=1000,
            C=1.0,
            solver="lbfgs",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X_train_smote, y_train_smote)

        # Evaluate
        y_pred = model.predict(X_test_tfidf)
        macro_f1 = f1_score(y_test, y_pred, average="macro")

        models[axis_name] = model
        metadata["scores"][axis_name] = macro_f1

        print(f"  {axis_name} ({label_0}/{label_1}): macro-F1 = {macro_f1:.4f}")

    # -- Save to disk ---------------------------------------------------------
    os.makedirs(MODEL_DIR, exist_ok=True)

    with open(TFIDF_PATH, "wb") as f:
        pickle.dump(tfidf, f)

    with open(MODELS_PATH, "wb") as f:
        pickle.dump(models, f)

    with open(METADATA_PATH, "wb") as f:
        pickle.dump(metadata, f)

    avg_f1 = np.mean(list(metadata["scores"].values()))
    print(f"\n  Average macro-F1: {avg_f1:.4f}")
    print(f"  Models saved to: {MODEL_DIR}/")
    print("=" * 60)

    return tfidf, models

## 2. LOAD MODELS


In [5]:
def load_models():
    """
    Load the saved TF-IDF vectorizer and 4 LR models from disk.

    If models haven't been trained yet, trains and saves them first.

    Returns
    -------
    tfidf : TfidfVectorizer
        Fitted vectorizer.
    models : dict
        {axis_name: fitted LogisticRegression model}
    """
    # Check if all required files exist
    if (os.path.exists(TFIDF_PATH)
            and os.path.exists(MODELS_PATH)
            and os.path.exists(METADATA_PATH)):
        print("  Loading saved models from disk ...")

        with open(TFIDF_PATH, "rb") as f:
            tfidf = pickle.load(f)
        with open(MODELS_PATH, "rb") as f:
            models = pickle.load(f)
        with open(METADATA_PATH, "rb") as f:
            metadata = pickle.load(f)

        # Print saved scores for reference
        for axis_name, score in metadata["scores"].items():
            print(f"    {axis_name}: macro-F1 = {score:.4f}")
        avg_f1 = np.mean(list(metadata["scores"].values()))
        print(f"    Average macro-F1: {avg_f1:.4f}")
        print("  Models loaded successfully.\n")

        return tfidf, models

    else:
        print("  No saved models found. Training from scratch ...\n")
        return train_and_save_models()

## 3. CORE INFERENCE FUNCTION


In [6]:
def predict_mbti(text: str, tfidf=None, models=None, verbose=True):
    """
    Predict the MBTI personality type from a text sample.

    This is the main inference function. Given a raw string (a comment,
    paragraph, or essay), it:
        1. Cleans the text using Phase 2's preprocessing pipeline
        2. Vectorizes it using the fitted TF-IDF vectorizer
        3. Passes it through 4 binary classifiers (one per MBTI axis)
        4. Concatenates the 4 predictions into a 4-letter MBTI type

    Parameters
    ----------
    text : str
        Raw text input (comment, paragraph, essay, etc.)
    tfidf : TfidfVectorizer, optional
        Pre-loaded vectorizer. If None, loads from disk.
    models : dict, optional
        Pre-loaded models. If None, loads from disk.
    verbose : bool
        If True, prints the prediction with details.

    Returns
    -------
    dict with keys:
        'type'        : str  — 4-letter MBTI type (e.g., "INFP")
        'description' : str  — Brief description of the type
        'axes'        : dict — Per-axis predictions with confidence
        'clean_text'  : str  — The preprocessed text (for debugging)

    Example
    -------
    >>> result = predict_mbti("I love spending time alone reading books ...")
    >>> print(result['type'])  # "INFP"
    """
    # -- Step 0: Load models if not provided ----------------------------------
    if tfidf is None or models is None:
        tfidf, models = load_models()

    # -- Step 1: Preprocess the raw text --------------------------------------
    # Uses the EXACT same cleaning pipeline from Phase 2:
    #   lowercase → remove URLs → remove MBTI tokens → remove non-alpha
    #   → remove stopwords → lemmatize → final MBTI safety net
    clean_text = preprocess_text(text)

    if not clean_text.strip():
        return {
            "type": "????",
            "description": "Could not extract enough meaningful text. Try a longer sample.",
            "axes": {},
            "clean_text": "",
        }

    # -- Step 2: Vectorize using the fitted TF-IDF ----------------------------
    # transform() expects a list/array of documents, so we wrap in a list
    text_vector = tfidf.transform([clean_text])

    # -- Step 3: Predict each MBTI axis ---------------------------------------
    mbti_letters = []
    axes_detail = {}

    for axis_name, pos, label_0, label_1 in MBTI_AXES:
        model = models[axis_name]

        # Get the binary prediction (0 or 1)
        prediction = model.predict(text_vector)[0]

        # Get probability estimates for confidence scoring
        # predict_proba returns [[P(class=0), P(class=1)]]
        probas = model.predict_proba(text_vector)[0]
        confidence = max(probas)  # Confidence = probability of the chosen class

        # Map prediction to the MBTI letter
        predicted_letter = label_1 if prediction == 1 else label_0
        mbti_letters.append(predicted_letter)

        axes_detail[axis_name] = {
            "prediction": predicted_letter,
            "confidence": confidence,
            "label_0": label_0,
            "label_1": label_1,
            "prob_0": probas[0],
            "prob_1": probas[1],
        }

    # -- Step 4: Concatenate into 4-letter MBTI type --------------------------
    mbti_type = "".join(mbti_letters)
    description = MBTI_DESCRIPTIONS.get(mbti_type, "Unknown MBTI type")

    # -- Print results if verbose ---------------------------------------------
    if verbose:
        print("\n" + "=" * 60)
        print("  MBTI PREDICTION RESULT")
        print("=" * 60)
        print(f"\n  Predicted Type: {mbti_type}")
        print(f"  {description}")
        print(f"\n  Per-Axis Breakdown:")
        print(f"  {'Axis':<8s}  {'Prediction':<12s}  {'Confidence':<12s}  {'Details'}")
        print(f"  {'-'*8}  {'-'*12}  {'-'*12}  {'-'*30}")

        for axis_name, detail in axes_detail.items():
            bar_len = int(detail["confidence"] * 20)
            bar = "#" * bar_len + "." * (20 - bar_len)
            print(
                f"  {axis_name:<8s}  {detail['prediction']:<12s}  "
                f"{detail['confidence']:<12.1%}  "
                f"{detail['label_0']}={detail['prob_0']:.3f} | "
                f"{detail['label_1']}={detail['prob_1']:.3f}  {bar}"
            )

        print(f"\n  Cleaned text preview: \"{clean_text[:120]}...\"")
        print("=" * 60)

    return {
        "type": mbti_type,
        "description": description,
        "axes": axes_detail,
        "clean_text": clean_text,
    }

## 5. MAIN


In [7]:
def main():
    """
    Execute the Phase 5 inference pipeline.

    Workflow:
        1. Load (or train+save) the models
        2. Run a few demo predictions
        3. Launch the Gradio web UI
    """
    print("\n" + "=" * 60)
    print("  PHASE 5: INFERENCE PIPELINE & WEB UI")
    print("=" * 60 + "\n")

    # -- Step 1: Load or train models ----------------------------------------
    tfidf, models = load_models()

    # -- Step 2: Demo predictions (showcase the predict_mbti function) --------
    print("\n" + "-" * 60)
    print("  DEMO PREDICTIONS")
    print("-" * 60)

    demo_texts = [
        (
            "Introvert + iNtuitive + Feeling + Perceiving sample",
            "I love spending time alone reading books and writing poetry. "
            "I often get lost in my imagination, thinking about abstract concepts "
            "and possibilities. I care deeply about how my decisions affect others, "
            "and I prefer to keep my options open rather than stick to a rigid plan."
        ),
        (
            "Extravert + Sensing + Thinking + Judging sample",
            "I'm a practical, no-nonsense kind of person. I love organising events "
            "and leading teams to achieve concrete goals. I make decisions based on "
            "logic and facts, not feelings. I enjoy being around lots of people and "
            "thrive in structured environments with clear deadlines."
        ),
    ]

    for label, text in demo_texts:
        print(f"\n  >> Demo: {label}")
        result = predict_mbti(text, tfidf=tfidf, models=models, verbose=True)

    # -- Step 3: Launch Gradio UI --------------------------------------------
    print("\n" + "-" * 60)
    print("  Starting Gradio Web Interface ...")
    print("-" * 60)
    launch_gradio_ui(tfidf, models)

## Interactive Prediction

Use the `predict_mbti()` function to predict MBTI type from any text.


In [8]:
# Load models
tfidf, models = load_models()

# --- Try your own text! Change the text below and re-run this cell ---
sample_text = """
I love spending time alone reading books and thinking about abstract concepts.
I care deeply about how my decisions affect others and I prefer to keep my
options open rather than stick to a rigid plan. Large parties drain me but
I enjoy deep one-on-one conversations about the meaning of life.
"""

result = predict_mbti(sample_text, tfidf=tfidf, models=models, verbose=True)
print(f"\nFinal prediction: {result['type']}")

  Loading saved models from disk ...
    E_I: macro-F1 = 0.6727
    S_N: macro-F1 = 0.6409
    T_F: macro-F1 = 0.7924
    J_P: macro-F1 = 0.6256
    Average macro-F1: 0.6829
  Models loaded successfully.




  MBTI PREDICTION RESULT

  Predicted Type: INTJ
  The Architect - Imaginative and strategic thinkers, with a plan for everything.

  Per-Axis Breakdown:
  Axis      Prediction    Confidence    Details
  --------  ------------  ------------  ------------------------------
  E_I       I             83.7%         E=0.163 | I=0.837  ################....
  S_N       N             62.3%         S=0.377 | N=0.623  ############........
  T_F       T             68.1%         T=0.681 | F=0.319  #############.......
  J_P       J             58.0%         J=0.580 | P=0.420  ###########.........

  Cleaned text preview: "love spending time alone reading book thinking abstract concept care deeply decision affect others prefer keep option op..."

Final prediction: INTJ


In [9]:
# --- Another example ---
sample_text_2 = """
I'm a practical, no-nonsense kind of person. I love organising events and
leading teams to achieve concrete goals. I make decisions based on logic and
facts, not feelings. I enjoy being around lots of people and thrive in
structured environments with clear deadlines.
"""

result2 = predict_mbti(sample_text_2, tfidf=tfidf, models=models, verbose=True)
print(f"\nFinal prediction: {result2['type']}")


  MBTI PREDICTION RESULT

  Predicted Type: ISTP
  The Virtuoso - Bold and practical experimenters, masters of all kinds of tools.

  Per-Axis Breakdown:
  Axis      Prediction    Confidence    Details
  --------  ------------  ------------  ------------------------------
  E_I       I             62.3%         E=0.377 | I=0.623  ############........
  S_N       S             60.2%         S=0.602 | N=0.398  ############........
  T_F       T             69.1%         T=0.691 | F=0.309  #############.......
  J_P       P             61.1%         J=0.389 | P=0.611  ############........

  Cleaned text preview: "practical nonsense kind person love organising event leading team achieve concrete goal make decision based logic fact f..."

Final prediction: ISTP


## Web UI (Gradio)

To launch the interactive Gradio web interface, run the following in a terminal:

```bash
python phase5_inference.py
```

Then open http://localhost:7860 in your browser.
